# Agentic AI Demo - Project Scaffolding Agent with Gemini

An **AI-powered project scaffolding agent** built with **Google Gemini 2.5 Flash**. Give it a project style and a topic, and it autonomously generates a complete folder structure with real, functional starter code - then executes the actions to scaffold everything on disk.

### What makes this Agentic AI?
This agent operates in a **true agentic loop** - not a single-shot prompt-and-execute:

- **Observe → Think → Act → Observe loop** - The agent runs iteratively. Each step it observes what's been done, decides the next action, executes it, sees the result, and repeats until done.
- **Tool use** - The LLM generates structured function calls (`create_folder`, `create_file`, `write_to_file`) that are executed on your filesystem
- **Autonomy** - Given only a style and topic, the agent independently decides the folder structure, file names, dependencies, and content - no human in the loop
- **Self-correction** - If an action fails, the error is fed back to the agent, which can adapt and retry
- **Structured output** - Each step returns machine-readable JSON executed via `eval()` with sandboxed globals
- **Memory** - Full conversation history is maintained across steps so the agent knows what it already created

> **TL;DR:** A regular AI *tells* you what to do. An agentic AI *does* it - step by step, with feedback.

</br>

> **Security** - Actions are executed via `eval()` with a **sandboxed `safe_globals`** dict that exposes only `create_folder`, `create_file`, and `write_to_file` - no access to `os`, `sys`, `subprocess`, builtins, or any other dangerous functions.

### How it works
1. Pick a **project style** and describe a **topic**
2. The agent enters a **loop** - each iteration it asks Gemini for the **next action**
3. Gemini returns a single action (create a folder, create a file, write content)
4. The action is **executed** and the **result** (success or error) is fed back
5. The agent **continues** until Gemini signals it's done - or a max-step limit is hit

### Supported Styles
`rest-api` · `cli-tool` · `web-app` · `fullstack` · `data-science` · `ml-pipeline` · `python-package` · `chrome-extension` · `discord-bot` · `mobile-app` · `microservice` · `static-site` · `game` · `vscode-extension` · `terraform`

In [1]:
!pip install -q -U google-genai python-dotenv

import os
from dotenv import load_dotenv
from google import genai
import json

load_dotenv()  # reads API key from .env file

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def create_folder(folder_path): os.makedirs(folder_path, exist_ok=True)
def create_file(file_path): open(file_path, "a").close()
def write_to_file(file_path, file_content): open(file_path, "w").write(file_content)

In [2]:
# Supported project styles
PROJECT_STYLES = {
    "rest-api":          "REST API backend (Flask/FastAPI) with routes, models, middleware",
    "cli-tool":          "Command-line tool with argument parsing and subcommands",
    "web-app":           "Frontend web app (HTML/CSS/JS or React/Vue)",
    "fullstack":         "Full-stack app with frontend + backend + database",
    "data-science":      "Data science project with notebooks, data, and scripts",
    "ml-pipeline":       "Machine learning pipeline with training, evaluation, and inference",
    "python-package":    "Publishable Python package with setup.py, tests, and docs",
    "chrome-extension":  "Chrome browser extension with manifest, popup, and background scripts",
    "discord-bot":       "Discord bot with commands, events, and cogs",
    "mobile-app":        "Mobile app (React Native / Flutter) with screens and navigation",
    "microservice":      "Microservice with Docker, health checks, and message queue support",
    "static-site":       "Static site / blog with templates and content",
    "game":              "Simple game project (Pygame / JS Canvas) with assets and scenes",
    "vscode-extension":  "VS Code extension with commands, views, and settings",
    "terraform":         "Infrastructure-as-Code project with Terraform modules and envs",
}

print("Supported project styles:\n")
for key, desc in PROJECT_STYLES.items():
    print(f"  • {key:20s} → {desc}")

Supported project styles:

  • rest-api             → REST API backend (Flask/FastAPI) with routes, models, middleware
  • cli-tool             → Command-line tool with argument parsing and subcommands
  • web-app              → Frontend web app (HTML/CSS/JS or React/Vue)
  • fullstack            → Full-stack app with frontend + backend + database
  • data-science         → Data science project with notebooks, data, and scripts
  • ml-pipeline          → Machine learning pipeline with training, evaluation, and inference
  • python-package       → Publishable Python package with setup.py, tests, and docs
  • chrome-extension     → Chrome browser extension with manifest, popup, and background scripts
  • discord-bot          → Discord bot with commands, events, and cogs
  • mobile-app           → Mobile app (React Native / Flutter) with screens and navigation
  • microservice         → Microservice with Docker, health checks, and message queue support
  • static-site          → Static si

In [3]:
SYSTEM_PROMPT = """You are an expert software architect agent. You scaffold projects ONE STEP AT A TIME.

You have access to these tools:
  create_folder(path)
  create_file(path)
  write_to_file(path, content)

On each turn you will receive the current state (what has been created so far, and the result of the last action).
You must respond with ONLY valid JSON in one of two formats:

1. To perform an action:
{
    "status": "next",
    "thought": "Brief reasoning about what to do next and why",
    "action": "create_folder('my-project/src')"
}

2. When the project is complete:
{
    "status": "done",
    "thought": "Summary of what was built",
    "project_name": "my-project"
}

Rules:
- Perform exactly ONE action per turn
- In write_to_file, ALWAYS wrap content in TRIPLE QUOTES: write_to_file('path', '''content''')
- Start by creating the root project folder, then build out from there
- Include ALL necessary config files (.gitignore, Dockerfile, package.json, etc.)
- Write REAL, functional starter code - not just placeholders
- Include a README with setup instructions
- Include requirements.txt or package.json with actual dependencies
- Use best practices for the given project style
- Signal "done" only after ALL files are created and written
"""

In [4]:
import ast

MAX_STEPS = 30  # safety limit

# only expose the allowed functions to eval
safe_globals = {
    "create_folder": create_folder,
    "create_file": create_file,
    "write_to_file": write_to_file,
}
# Parse LLM response text into a dict.  
def parse_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]
    try:
        return json.loads(text, strict=False)
    except json.JSONDecodeError:
        return ast.literal_eval(text)

# Agentic loop: ask Gemini for one action at a time, execute, feed result back, repeat.    
def scaffold_project(style: str, topic: str):
    if style not in PROJECT_STYLES:
        print(f"Unknown style '{style}'. Choose from: {', '.join(PROJECT_STYLES.keys())}")
        return

    # cd to output directory
    OUTPUT_DIR = os.environ["OUTPUT_DIR"]
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.chdir(OUTPUT_DIR)

    # conversation history (the agent's memory)
    messages = [
        f"""Create a **{style}** project about: "{topic}"
        Style description: {PROJECT_STYLES[style]}
        Scaffold the project step by step. Start now."""
    ]

    print(f"Agent started: {style} project - \"{topic}\"")
    print(f"Output dir: {OUTPUT_DIR}\n")

    # ── THINK: ask Gemini for next action ──
    for step in range(1, MAX_STEPS + 1):
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config={
                "system_instruction": SYSTEM_PROMPT,
                "temperature": 0.7,
            }
        )
        reply = response.text
        messages.append(reply)  # remember AI's response

        # ── PARSE ──
        try:
            step_data = parse_response(reply)
        except Exception as e:
            feedback = f"ERROR parsing your response: {e}. Please respond with valid JSON."
            print(f"  Step {step}: Parse error - retrying")
            messages.append(feedback)
            continue

        thought = step_data.get("thought", "")
        status  = step_data.get("status", "next")

        # ── DONE? ──
        if status == "done":
            project_name = step_data.get("project_name", "unknown")
            print(f"\nAgent finished in {step} steps!")
            print(f"   Thinking {thought}")
            print(f"   Project at: {os.path.join(OUTPUT_DIR, project_name)}")
            return

        # ── ACT: execute the action ──
        action = step_data.get("action", "")
        if not action:
            feedback = "ERROR: no 'action' field in your response. Include an action."
            messages.append(feedback)
            continue

        try:
            if action.startswith("write_to_file"):
                print(f"  Step {step}: {thought[:80]}…" if len(thought) > 80 else f"  Step {step}: {thought}")
                print(f"           write_to_file")
            else:
                print(f"  Step {step}: {thought[:80]}…" if len(thought) > 80 else f"  Step {step}: {thought}")
                print(f"           {action}")
            eval(action, safe_globals)
            feedback = f"SUCCESS: {action.split('(')[0]} executed successfully. Think about the next steps"
        except Exception as e:
            feedback = f"ERROR executing action: {e}. Please fix and try again."
            print(f"           {e}")

        # ── OBSERVE: feed result back to agent ──
        messages.append(feedback)

    print(f"\nAgent hit max steps ({MAX_STEPS}) without finishing.")

In [5]:
# Run the agent
scaffold_project("rest-api", "a bookstore inventory management system")

Agent started: rest-api project - "a bookstore inventory management system"
Output dir: /Users/arantarokade/Documents/speedrun-gen-ai/output



  Step 1: Starting the project by creating the root folder `bookstore-api`.
           create_folder('bookstore-api')
  Step 2: The root folder `bookstore-api` has been created. I'll use FastAPI for this proj…
           create_folder('bookstore-api/app')
  Step 3: The `app` folder is created. Now I need to create the main application file `mai…
           create_file('bookstore-api/app/main.py')
  Step 4: The `main.py` file is ready. I will now write the initial FastAPI setup into `ma…
           write_to_file
  Step 5: The `main.py` is initialized. Now, I'll create a `models` directory within the `…
           create_folder('bookstore-api/app/models')
  Step 6: The `models` folder is created. Now I'll create `book.py` inside `app/models` to…
           create_file('bookstore-api/app/models/book.py')
  Step 7: The `book.py` file is created. Now I will write the Pydantic model for a `Book` …
           write_to_file


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 6.012578975s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}

In [ ]:
# ── More examples (uncomment any to try) ──────────────────────────────

# scaffold_project("cli-tool",          "a file duplicate finder")
# scaffold_project("fullstack",         "a todo app with authentication")
# scaffold_project("ml-pipeline",       "sentiment analysis on product reviews")
# scaffold_project("discord-bot",       "a music trivia quiz bot")
# scaffold_project("chrome-extension",  "a website reading time estimator")
# scaffold_project("microservice",      "a URL shortener service")
# scaffold_project("data-science",      "analyzing global CO2 emissions")
# scaffold_project("python-package",    "a markdown-to-HTML converter library")
# scaffold_project("terraform",         "AWS ECS deployment with ALB")
# scaffold_project("game",              "a snake game with power-ups")
# scaffold_project("vscode-extension",  "a code snippet manager")
# scaffold_project("mobile-app",        "a habit tracker with streaks")
# scaffold_project("static-site",       "a personal developer portfolio")